In [1]:
!pip install groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import sqlite3
from groq import Groq
import os

In [3]:
from google.colab import files

uploaded = files.upload()

Saving student_performance.csv to student_performance (1).csv


In [4]:
df = pd.read_csv("student_performance.csv")

df.head()

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [5]:
conn = sqlite3.connect("college.db")

df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

print("Database Created")

Database Created


In [7]:
from groq import Groq
import os

os.environ["GROQ_API_KEY"] = ""

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

MODEL = "llama-3.1-8b-instant"

In [8]:
query = """
SELECT * FROM students LIMIT 5
"""

pd.read_sql(query, conn)

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [9]:
print(df.columns)

Index(['student_id', 'name', 'age', 'gender', 'department', 'semester',
       'math_score', 'science_score', 'english_score', 'programming_score',
       'attendance_percentage', 'city', 'admission_year'],
      dtype='object')


In [10]:
query = """
SELECT COUNT(*) AS total_students
FROM students
"""

pd.read_sql(query, conn)

,total_students
0,30


In [11]:
query = """
SELECT *
FROM students
WHERE gender='F'
"""

pd.read_sql(query, conn)

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year


In [12]:
def get_schema(conn):
    cursor = conn.cursor()

    cursor.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name='students'"
    )

    return cursor.fetchone()[0]

print(get_schema(conn))

CREATE TABLE "students" (
"student_id" INTEGER,
  "name" TEXT,
  "age" INTEGER,
  "gender" TEXT,
  "department" TEXT,
  "semester" INTEGER,
  "math_score" INTEGER,
  "science_score" INTEGER,
  "english_score" INTEGER,
  "programming_score" INTEGER,
  "attendance_percentage" INTEGER,
  "city" TEXT,
  "admission_year" INTEGER
)


In [14]:
def generate_sql(question, schema, client, model):

    prompt = f"""
    You are an SQL expert.

    Database Schema:
    {schema}

    Convert the following question into SQL.

    Question:
    {question}

    Return only SQL.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content

In [15]:
schema = get_schema(conn)

sql = generate_sql(
    "Show all female students",
    schema,
    client,
    MODEL
)

print(sql)

SELECT * FROM students WHERE gender = 'F'


In [16]:
def execute_sql(sql_query, conn):
    try:
        df = pd.read_sql(sql_query, conn)
        return df, None
    except Exception as e:
        return None, str(e)

In [17]:
sql = generate_sql(
    "Show all female students",
    schema,
    client,
    MODEL
)

results, error = execute_sql(sql, conn)

if error:
    print(error)
else:
    print(results)

Execution failed on sql '```sql
SELECT * 
FROM students 
WHERE gender = 'F';
```': near "```sql
SELECT * 
FROM students 
WHERE gender = 'F';
```": syntax error


In [18]:
def ai_data_analyst(question):

    schema = get_schema(conn)

    sql = generate_sql(
        question,
        schema,
        client,
        MODEL
    )

    print("Generated SQL:")
    print(sql)

    results, error = execute_sql(
        sql,
        conn
    )

    if error:
        print(error)
        return

    print("\nResults:")
    print(results)

In [19]:
ai_data_analyst(
    "Show all female students"
)

Generated SQL:
```sql
SELECT * FROM students WHERE gender = "female";
```
Execution failed on sql '```sql
SELECT * FROM students WHERE gender = "female";
```': near "```sql
SELECT * FROM students WHERE gender = "female";
```": syntax error


In [20]:
ai_data_analyst(
    "Show students scoring above 85 in Science"
)

Generated SQL:
```sql
SELECT *
FROM students
WHERE science_score > 85;
```
Execution failed on sql '```sql
SELECT *
FROM students
WHERE science_score > 85;
```': near "```sql
SELECT *
FROM students
WHERE science_score > 85;
```": syntax error


In [21]:
ai_data_analyst(
    "How many male students are there?"
)

Generated SQL:
SELECT COUNT(*) 
FROM students 
WHERE gender = 'Male'

Results:
   COUNT(*)
0        15


In [22]:
ai_data_analyst("Show all female students")

Generated SQL:
SELECT * FROM students WHERE gender = 'female';

Results:
Empty DataFrame
Columns: [student_id, name, age, gender, department, semester, math_score, science_score, english_score, programming_score, attendance_percentage, city, admission_year]
Index: []


In [23]:
ai_data_analyst("How many male students are there?")

Generated SQL:
```sql
SELECT COUNT(*) 
FROM students 
WHERE gender = 'male';
```
Execution failed on sql '```sql
SELECT COUNT(*) 
FROM students 
WHERE gender = 'male';
```': near "```sql
SELECT COUNT(*) 
FROM students 
WHERE gender = 'male';
```": syntax error


In [24]:
ai_data_analyst("Show students scoring above 85 in Science")

Generated SQL:
```sql
SELECT *
FROM students
WHERE science_score > 85;
```
Execution failed on sql '```sql
SELECT *
FROM students
WHERE science_score > 85;
```': near "```sql
SELECT *
FROM students
WHERE science_score > 85;
```": syntax error
